In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import MinMaxScaler

In [2]:
bronx_matched = pd.read_csv('../data/processed/bronx_matched_clustered.csv')
bronx_matched['crash_date'] = pd.to_datetime(bronx_matched['crash_date'])

cluster_summary = pd.read_csv('../data/processed/bronx_cluster_summary.csv')

In [3]:
'cluster' in bronx_matched.columns

True

In [4]:
bronx_matched[['collision_id', 'cluster', 'number_of_persons_injured', 'number_of_persons_killed']].head()

,collision_id,cluster,number_of_persons_injured,number_of_persons_killed
0,00001c353265e15ae44bdd8eb2a429c4,0,0.0,0.0
1,0000ce397c063570a0da1420fa22a955,1,0.0,0.0
2,0000e467ad5c1e6e09bfc6135f12802d,2,0.0,0.0
3,00012c6e1b9be0fa2b058b3f1bd05e42,3,0.0,0.0
4,00015fe56e997dcd5d42470bcaf91a3f,4,1.0,0.0


In [5]:
bronx_matched[['number_of_persons_injured', 'number_of_persons_killed']].sum()

number_of_persons_injured    118064.0
number_of_persons_killed        566.0
dtype: float64

In [6]:
severity_by_cluster = bronx_matched[bronx_matched['cluster'] != -1].groupby('cluster').agg(
    total_injured=('number_of_persons_injured', 'sum'),
    total_killed=('number_of_persons_killed', 'sum'),
    fatal_crash_count=('number_of_persons_killed', lambda x: (x > 0).sum()),
    injury_rate=('number_of_persons_injured', lambda x: (x > 0).mean() * 100)
).reset_index()

severity_by_cluster.head()

,cluster,total_injured,total_killed,fatal_crash_count,injury_rate
0,0,241.0,0.0,0,34.341637
1,1,3.0,0.0,0,17.647059
2,2,219.0,2.0,2,23.903177
3,3,24.0,1.0,1,24.675325
4,4,22.0,0.0,0,30.000000


In [7]:
cluster_summary = cluster_summary.merge(severity_by_cluster, on='cluster', how='left')
cluster_summary.head()

,cluster,crash_count,center_lat,center_lon,top_street,total_injured,total_killed,fatal_crash_count,injury_rate
0,0,562,40.861696,-73.891380,EAST FORDHAM ROAD,241.0,0.0,0,34.341637
1,1,17,40.853036,-73.918260,SEDGWICK AVENUE,3.0,0.0,0,17.647059
2,2,661,40.878643,-73.871601,EAST GUN HILL ROAD,219.0,2.0,2,23.903177
3,3,77,40.830258,-73.914854,EAST 166 STREET,24.0,1.0,1,24.675325
4,4,50,40.841511,-73.861768,EAST TREMONT AVENUE,22.0,0.0,0,30.000000


In [8]:
cluster_summary.shape

(5324, 9)

In [9]:
scaler = MinMaxScaler()
cluster_summary[['crash_count_norm', 'injured_norm', 'killed_norm']] = scaler.fit_transform(
    cluster_summary[['crash_count', 'total_injured', 'total_killed']]
)

crash_weight = 1
injured_weight = 1
killed_weight = 2

cluster_summary['danger_score'] = (
    crash_weight * cluster_summary['crash_count_norm']
    + injured_weight * cluster_summary['injured_norm']
    + killed_weight * cluster_summary['killed_norm']
)

cluster_summary.sort_values('danger_score', ascending=False).head(10)

,cluster,crash_count,center_lat,center_lon,top_street,total_injured,total_killed,fatal_crash_count,injury_rate,crash_count_norm,injured_norm,killed_norm,danger_score
323,323,1383,40.820212,-73.890665,BRUCKNER BOULEVARD,563.0,1.0,1,27.621114,1.000000,0.893651,0.25,2.393651
810,810,227,40.809109,-73.922889,EAST 138 STREET,118.0,4.0,4,40.088106,0.163531,0.187302,1.00,2.350833
932,932,486,40.845172,-73.911192,CROSS BRONX EXPY,257.0,3.0,2,30.246914,0.350941,0.407937,0.75,2.258877
352,352,177,40.826571,-73.931075,MAJOR DEEGAN EXPRESSWAY,79.0,4.0,4,27.683616,0.127352,0.125397,1.00,2.252748
2046,2046,69,40.865464,-73.865395,ALLERTON AVENUE,33.0,4.0,2,30.434783,0.049204,0.052381,1.00,2.101585
125,125,806,40.838008,-73.873283,BRONX RIVER PARKWAY,321.0,2.0,2,25.558313,0.582489,0.509524,0.50,2.092013
94,94,54,40.811314,-73.900765,BRUCKNER BOULEVARD,20.0,4.0,3,25.925926,0.038350,0.031746,1.00,2.070096
51,51,1310,40.861863,-73.912783,WEST FORDHAM ROAD,630.0,0.0,0,30.381679,0.947178,1.000000,0.00,1.947178
29,29,734,40.804727,-73.912500,BRUCKNER BOULEVARD,236.0,2.0,2,23.433243,0.530391,0.374603,0.50,1.904994
187,187,750,40.804673,-73.922346,BRUCKNER BOULEVARD,179.0,2.0,2,16.800000,0.541968,0.284127,0.50,1.826095


In [10]:
# spot check the crashes in the most dangerous hotspot
top_cluster = cluster_summary.sort_values('danger_score', ascending=False)['cluster'].iloc[0]
bronx_matched[bronx_matched['cluster'] == top_cluster][['collision_id', 'crash_date', 'number_of_persons_injured', 'number_of_persons_killed']].sort_values('number_of_persons_killed', ascending=False).head(10)

,collision_id,crash_date,number_of_persons_injured,number_of_persons_killed
142503,77534c73ee484bf2d45d568b9c2c7f50,2017-04-23,0.0,1.0
424,005c538b31854ebbf70611801715aaca,2022-03-13,0.0,0.0
199655,a7227c0d8081f8bd71c52d1bf316ca0f,2019-06-21,3.0,0.0
202864,a9c54ddde1d423d49d54e08aa5bd78d3,2013-07-22,0.0,0.0
202837,a9beec681d4c4f56176b19c4fd34ac9b,2015-10-05,0.0,0.0
202782,a9b52ceef07a4478e10b5a3dff795a70,2013-05-22,0.0,0.0
202738,a9ab2d446d93b0d082f9d3b90f551791,2020-08-08,2.0,0.0
202725,a9a7db2c50ec7baad6aac7df07ce380d,2017-11-02,1.0,0.0
202609,a98ecd9484a09454b45599d66fbc4d87,2020-07-12,0.0,0.0
202558,a98621e6eb07fee9e21ee724e4d8f713,2021-08-05,0.0,0.0


In [11]:
# NYPD sometimes typed a full address into the street fields ("1961 MAPES AVENUE")
# strip leading house numbers so addresses count as their street, but keep
# numbered streets like "3 AVENUE" (nothing after the number but the street type)
# newer records also abbreviate ("CROSS BRONX EXPY", "E 149 ST"), so spell those out to match the older ones

# street types only get expanded at the end, so "ST ANNS AVENUE" and "DR MARTIN L KING JR BLVD" keep their first word
STREET_TYPES = {
    'AVE': 'AVENUE', 'AV': 'AVENUE', 'ST': 'STREET', 'RD': 'ROAD', 'DR': 'DRIVE',
    'BLVD': 'BOULEVARD', 'PKWY': 'PARKWAY', 'PL': 'PLACE', 'EXPY': 'EXPRESSWAY',
    'EXPWY': 'EXPRESSWAY', 'LN': 'LANE', 'TER': 'TERRACE', 'CT': 'COURT', 'SQ': 'SQUARE',
    'CONC': 'CONCOURSE',
}
# directions get expanded as the first word ("E 149 ST") or the last word ("PELHAM PKWY S")
DIRECTIONS = {'E': 'EAST', 'W': 'WEST', 'N': 'NORTH', 'S': 'SOUTH'}
# these never mean anything else, so they get spelled out anywhere in the name ("CROSS BRONX EXPY RAMP")
ALWAYS_EXPAND = {'EXPY', 'EXPWY', 'PKWY', 'BLVD'}

def expand_abbreviations(name):
    words = [STREET_TYPES[w] if w in ALWAYS_EXPAND else w for w in name.split()]
    if len(words) > 1 and words[0] in DIRECTIONS:
        words[0] = DIRECTIONS[words[0]]
    # when a street ends in a direction, the street type is the word before it
    if len(words) > 2 and words[-1] in DIRECTIONS:
        words[-1] = DIRECTIONS[words[-1]]
        type_pos = -2
    else:
        type_pos = -1
    if words[type_pos] in STREET_TYPES:
        words[type_pos] = STREET_TYPES[words[type_pos]]
    return ' '.join(words)

def clean_street(col):
    col = col.str.strip().str.upper().str.replace(r'\s+', ' ', regex=True)
    col = col.str.replace(r'\b(\d+)(ST|ND|RD|TH)\b', r'\1', regex=True)  # "149TH" -> "149" like the older records
    col = col.str.replace(r'^\d+[A-Z]?\s+(?=\S+\s+\S)', '', regex=True)
    col = col.replace('', pd.NA)
    return col.map(expand_abbreviations, na_action='ignore')

# off_street_name only gets filled when on_street_name is blank, so use it as a fallback
bronx_matched['on_street_clean'] = clean_street(bronx_matched['on_street_name'].fillna(bronx_matched['off_street_name']))
bronx_matched['cross_street_clean'] = clean_street(bronx_matched['cross_street_name'])

# recompute top_street from the cleaned names
clustered = bronx_matched[bronx_matched['cluster'] != -1]
top_street = clustered.groupby('cluster')['on_street_clean'].agg(
    lambda x: x.mode()[0] if not x.mode().empty else None
).reset_index(name='top_street')

cluster_summary = cluster_summary.drop(columns='top_street').merge(top_street, on='cluster', how='left')

In [12]:
# second street for each hotspot, so labels read "Road & Cross Street"
# pool on_street and cross_street since NYPD sometimes swaps which one goes where
def second_street(group, top):
    names = pd.concat([group['on_street_clean'], group['cross_street_clean']]).dropna()
    names = names[names != top]
    return names.mode()[0] if not names.mode().empty else None

clustered = bronx_matched[bronx_matched['cluster'] != -1]
top_by_cluster = cluster_summary.set_index('cluster')['top_street']

# only pass the street columns so pandas doesn't also hand the cluster column to the function
cross_street = clustered.groupby('cluster')[['on_street_clean', 'cross_street_clean']].apply(
    lambda g: second_street(g, top_by_cluster.get(g.name))
).reset_index(name='cross_street')

cluster_summary = cluster_summary.merge(cross_street, on='cluster', how='left')

print(cluster_summary['top_street'].isna().sum(), 'with no top_street')
print((cluster_summary['top_street'].isna() & cluster_summary['cross_street'].isna()).sum(), 'with no name at all')
cluster_summary[['cluster', 'top_street', 'cross_street']].head(10)

1 with no top_street
1 with no name at all


,cluster,top_street,cross_street
0,0,EAST FORDHAM ROAD,WEBSTER AVENUE
1,1,SEDGWICK AVENUE,WEST TREMONT AVENUE
2,2,EAST GUN HILL ROAD,WEBSTER AVENUE
3,3,EAST 166 STREET,COLLEGE AVENUE
4,4,EAST TREMONT AVENUE,DOGWOOD DRIVE
5,5,EAST 149 STREET,BROOK AVENUE
6,6,JEROME AVENUE,GOBLE PLACE
7,7,WESTCHESTER AVENUE,WHITLOCK AVENUE
8,8,BRUCKNER EXPRESSWAY,None
9,9,BRONX WHITESTONE BRIDGE,None


In [13]:
# most common first and last words, to catch any abbreviation the lists missed
names = pd.concat([bronx_matched['on_street_clean'], bronx_matched['cross_street_clean']]).dropna()
print(names.str.split().str[-1].value_counts().head(30))
print(names.str.split().str[0].value_counts().head(15))

AVENUE        228407
STREET        115066
ROAD           41586
EXPRESSWAY     26990
BOULEVARD      22490
PARKWAY        11747
PLACE          11092
CONCOURSE       7286
BROADWAY        2908
EAST            2745
SOUTH           2160
NORTH           1647
RAMP            1258
BRIDGE           878
TERRACE          836
THRUWAY          747
WEST             742
LANE             712
DRIVE            693
LOOP             588
HIGHWAY          487
CRESCENT         325
APPROACH         209
OVAL             168
EXTENTION        166
CIRCLE           157
SQUARE           114
ESPLANADE         90
BRG               69
PARK              64
Name: count, dtype: int64
EAST           107290
WEST            21686
BRUCKNER        19560
CROSS           10902
MAJOR           10718
BRONX           10375
JEROME           8720
GRAND            8476
WHITE            8117
WESTCHESTER      8082
WEBSTER          7854
BOSTON           7673
MORRIS           6056
3                5962
SOUTHERN         5553
Name: count, d

In [14]:
# most common recorded cause, from the original causes rather than the grouped ones:
# "Other" bundles 54 causes, so as one bucket it outvotes any single real cause
def top_recorded_cause(x):
    recorded = x.dropna().str.strip()
    recorded = recorded[(recorded != 'Unspecified') & ~recorded.str.fullmatch(r'\d+')]  # skip the unmapped numeric codes
    if recorded.mode().empty:
        return 'Unspecified'  # no cause recorded for any crash here
    return recorded.mode()[0]

top_factor = clustered.groupby('cluster')['contributing_factor_vehicle_1'].agg(top_recorded_cause).reset_index(name='top_factor')

cluster_summary = cluster_summary.merge(top_factor, on='cluster', how='left')
cluster_summary

,cluster,crash_count,center_lat,center_lon,total_injured,total_killed,fatal_crash_count,injury_rate,crash_count_norm,injured_norm,killed_norm,danger_score,top_street,cross_street,top_factor
0,0,562,40.861696,-73.891380,241.0,0.0,0,34.341637,0.405933,0.382540,0.00,0.788473,EAST FORDHAM ROAD,WEBSTER AVENUE,Driver Inattention/Distraction
1,1,17,40.853036,-73.918260,3.0,0.0,0,17.647059,0.011577,0.004762,0.00,0.016339,SEDGWICK AVENUE,WEST TREMONT AVENUE,Driver Inattention/Distraction
2,2,661,40.878643,-73.871601,219.0,2.0,2,23.903177,0.477569,0.347619,0.50,1.825188,EAST GUN HILL ROAD,WEBSTER AVENUE,Driver Inattention/Distraction
3,3,77,40.830258,-73.914854,24.0,1.0,1,24.675325,0.054993,0.038095,0.25,0.593088,EAST 166 STREET,COLLEGE AVENUE,Driver Inattention/Distraction
4,4,50,40.841511,-73.861768,22.0,0.0,0,30.000000,0.035456,0.034921,0.00,0.070376,EAST TREMONT AVENUE,DOGWOOD DRIVE,Driver Inattention/Distraction
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5319,5319,12,40.842008,-73.869820,0.0,0.0,0,0.000000,0.007959,0.000000,0.00,0.007959,MELVILLE STREET,None,Driver Inattention/Distraction
5320,5320,10,40.849447,-73.856255,0.0,0.0,0,0.000000,0.006512,0.000000,0.00,0.006512,PAULDING AVENUE,None,Other Vehicular
5321,5321,10,40.813002,-73.886100,0.0,0.0,0,0.000000,0.006512,0.000000,0.00,0.006512,FAILE STREET,SPOFFORD AVENUE,Backing Unsafely
5322,5322,10,40.842032,-73.837073,4.0,0.0,0,30.000000,0.006512,0.006349,0.00,0.012862,ZULETTE AVENUE,ERICSON PLACE,Backing Unsafely


In [15]:
# final top 10 with cleaned names
cluster_summary.sort_values('danger_score', ascending=False)[
    ['top_street', 'cross_street', 'crash_count', 'total_injured', 'total_killed', 'danger_score']
].head(10)

,top_street,cross_street,crash_count,total_injured,total_killed,danger_score
323,BRUCKNER BOULEVARD,HUNTS POINT AVENUE,1383,563.0,1.0,2.393651
810,EAST 138 STREET,WILLIS AVENUE,227,118.0,4.0,2.350833
932,CROSS BRONX EXPRESSWAY,GRAND CONCOURSE,486,257.0,3.0,2.258877
352,MAJOR DEEGAN EXPRESSWAY,MACOMBS DAM BRIDGE,177,79.0,4.0,2.252748
2046,ALLERTON AVENUE,HOLLAND AVENUE,69,33.0,4.0,2.101585
125,BRONX RIVER PARKWAY,BRONX RIVER PARKWAY RAMP,806,321.0,2.0,2.092013
94,BRUCKNER BOULEVARD,AUSTIN PLACE,54,20.0,4.0,2.070096
51,WEST FORDHAM ROAD,MAJOR DEEGAN EXPRESSWAY,1310,630.0,0.0,1.947178
29,BRUCKNER BOULEVARD,EAST 138 STREET,734,236.0,2.0,1.904994
187,BRUCKNER BOULEVARD,BROWN PLACE,750,179.0,2.0,1.826095


In [16]:
cluster_summary.to_csv('../data/processed/bronx_cluster_summary_scored.csv', index=False)